# Step 2 on Kaggle: train FaST-ISL (resumes across sessions)

**Settings (right-hand panel) before running:**
- *Session options → Accelerator:* **GPU T4 x2** (one GPU is used)
- *Session options → Internet:* **On**
- *Input → Add Input → Your Work → Notebooks:* add the **output of the step-1 conversion notebook**
- **From the 2nd run on:** also add the **output of this notebook's previous version**
  (*Add Input → Your Work → Notebooks → this notebook*). That is how training resumes: the runs folder of
  the last session is copied back in. If Kaggle does not offer it, open the previous version's *Output* tab
  → **New Dataset**, and add that dataset instead.

Click **Save Version → Save & Run All (Commit)**. It runs in the background for up to ~11 hours, then stops
cleanly. Each new commit continues where the last one stopped: finished models are skipped and the
current one resumes from its last epoch. About 30 GPU hours per week are free.

Order: baseline and full model first (the paper's main result), then the ablations, then the
sentence-boundary detector and the continuous-signing evaluation.

In [ ]:
import time; NOTEBOOK_START = time.time()
BUDGET_H = 11.0   # Kaggle stops GPU sessions at 12 h; leave room to save the output
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /tmp/ISL && git clone -q https://github.com/sengarshivansh/ISL.git /tmp/ISL
!pip -q install sacrebleu rouge-score sentencepiece pyyaml
%cd /tmp/ISL

In [ ]:
import glob, os, shutil
# keypoint shards from the step-1 notebook
hits = glob.glob('/kaggle/input/**/shard_00000.npz', recursive=True)
assert hits, 'Add the output of the step-1 conversion notebook as an input (see the instructions above)'
DATA = os.path.dirname(hits[0])
print(len(glob.glob(f'{DATA}/shard_*.npz')), 'shards in', DATA)

# point every step at the Kaggle paths
cfg_path = 'configs/isign_base.yaml'
text = open(cfg_path).read().replace('/content/drive/MyDrive/ISL/isign_pose_keypoints', DATA)
open(cfg_path, 'w').write(text)

# runs from earlier sessions (output of a previous version of this notebook)
RUNS = '/kaggle/working/runs'
prev = [p for p in glob.glob('/kaggle/input/**/runs', recursive=True) if os.path.isdir(p)]
if prev and not os.path.exists(RUNS):
    shutil.copytree(prev[0], RUNS)
    print('resuming from', prev[0])
os.makedirs(RUNS, exist_ok=True)
for d in sorted(os.listdir(RUNS)):
    state = 'done' if os.path.exists(f'{RUNS}/{d}/results.json') else 'in progress'
    print(f'  {d}: {state}')

In [ ]:
import subprocess, time
def hours_left():
    return BUDGET_H - (time.time() - NOTEBOOK_START) / 3600

order = ['B0_body_hands', 'FaST', 'B1_speedaug', 'FaST_noaug', 'FaST_concat',
         'F_mouth', 'F_eyebrows', 'F_eyes', 'F_head']
for name in order:
    if os.path.exists(f'{RUNS}/{name}/results.json'):
        print('done:', name); continue
    if hours_left() < 1.0:
        print('not enough time left in this session: commit again to continue'); break
    print(f'=== training {name} ({hours_left():.1f} h left in this session)', flush=True)
    code = f'''
import yaml
from islt.train import load_config, deep_update, train
cfg = load_config("configs/isign_base.yaml")
deep_update(cfg, yaml.safe_load(open("configs/ablations/{name}.yaml")))
cfg["out_dir"] = "{RUNS}"
cfg["train"]["max_hours"] = {hours_left():.2f}
train(cfg)
'''
    subprocess.run(['python', '-c', code], check=True)
    if not os.path.exists(f'{RUNS}/{name}/results.json'):
        print(f'{name} paused at the time limit: commit again to continue'); break

In [ ]:
# sentence-boundary detector (about 20 min) and continuous-signing evaluation, once the main models exist
main_done = all(os.path.exists(f'{RUNS}/{n}/results.json') for n in ['B0_body_hands', 'FaST'])
if main_done and not os.path.exists(f'{RUNS}/boundary/boundary.pt') and hours_left() > 0.7:
    !python -m islt.boundary --config configs/isign_base.yaml --out {RUNS}/boundary --steps 6000 --device cuda
if main_done and os.path.exists(f'{RUNS}/boundary/boundary.pt'):
    for run in ['FaST', 'B0_body_hands']:
        if hours_left() < 0.5: break
        if not os.path.exists(f'{RUNS}/{run}/stream_results.json'):            # pause-based segmenter
            !python -m islt.eval_stream --config configs/isign_base.yaml --checkpoint {RUNS}/{run}/best.pt
        if not os.path.exists(f'{RUNS}/{run}/stream_results_learned.json'):    # learned boundary detector
            !python -m islt.eval_stream --config configs/isign_base.yaml --checkpoint {RUNS}/{run}/best.pt --boundary {RUNS}/boundary/boundary.pt

In [ ]:
!python -m islt.report {RUNS} --out /kaggle/working/results_table.tex
!ls -la {RUNS}/*/